# Paso 02 -- Escalon 2: dia completo, metricas agregadas

Corre el entorno (`SimuladorBarcosBergen`, mismo modelo de colas por par y
misma politica coordinada `asignar_flota` que el escalon 1) sobre el DIA
COMPLETO (6:00-24:00, 3 barcos, ~133 grupos / 1106 personas,
`output/escalon2/grupos.csv` del paso 00).

**Diferencias respecto al escalon 1 (`01_escalon_1_verificacion.ipynb`):**
- 540 pasos de 2 min (vs. 90 en el escalon 1) -- no se imprime un log de
  texto paso a paso (seria una pared de 540 lineas, no una verificacion a
  ojo util). El foco aqui son las metricas agregadas.
- **Sin animacion ni reproductor interactivo.** Un GIF de 540 frames pesa
  y tarda mucho mas que el de 90 frames del escalon 1, y la verificacion
  visual paso a paso ya se hizo en el escalon 1 -- no hace falta repetirla
  a esta escala. Si mas adelante hace falta ver la pelicula completa del
  dia, se puede agregar reusando `visualizacion.animar_corrida` /
  `reproductor_interactivo` tal cual estan en el escalon 1.
- Todo se guarda en su propia carpeta, `output/escalon2/`, en paralelo a
  `output/escalon1/`.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
BB_DIR = (BASE_DIR / "../bergen-boats").resolve()
sys.path.insert(0, str(BASE_DIR / "src"))

from env import SimuladorBarcosBergen
from politica_base import politica_base, asignar_flota
import metricas as met
import visualizacion as viz

cfg_sim = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))

nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

OUT_ESC = BASE_DIR / "output" / "escalon2"
OUT_ESC.mkdir(parents=True, exist_ok=True)
grupos_escalon2 = pd.read_csv(OUT_ESC / "grupos.csv")

cfg_escalon = cfg_sim["escalones"]["escalon_2"]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60


def construir_env():
    return SimuladorBarcosBergen(
        grupos_df=grupos_escalon2, matriz_tiempos=matriz_tiempos, nodos=nodos,
        num_barcos=cfg_escalon["num_barcos"], capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"],
        nodo_inicial=cfg_bb["flota"]["nodo_inicial"], paso_tiempo_min=cfg_sim["paso_tiempo_min"],
        hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )


n_pasos_esperados = int((hora_fin_min - hora_ini_min) / cfg_sim["paso_tiempo_min"])
print("Nodos:", nodos)
print(f"Barcos: {cfg_escalon['num_barcos']}, capacidad: {cfg_bb['flota']['capacidad_pasajeros']}")
print(f"Horizonte: {hora_ini_min}-{hora_fin_min} min, paso: {cfg_sim['paso_tiempo_min']} min -> {n_pasos_esperados} pasos")
print(f"Demanda: {len(grupos_escalon2)} grupos, {grupos_escalon2['tamano_grupo'].sum()} personas")


Nodos: ['kleppesto', 'laksevag', 'bryggen', 'sandviken']
Barcos: 3, capacidad: 20
Horizonte: 360-1440 min, paso: 2 min -> 540 pasos
Demanda: 133 grupos, 1106 personas


## Corrida completa (sin log paso a paso -- son 540 pasos)

In [2]:
env = construir_env()
obs, info = env.reset(seed=cfg_sim["semilla"])
reward_total = 0.0
paso = 0

while True:
    estado = info["_estado_obj"]
    libres = [b for b in estado.barcos if b.libre]
    decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
    accion = np.array([
        env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
        for b in estado.barcos
    ])
    obs, r, terminated, truncated, info = env.step(accion)
    reward_total += r
    paso += 1
    if paso % 90 == 0:  # un aviso de avance cada ~3h simuladas, no un log completo
        t = info["estado_dict"]["tiempo"]["minuto_del_dia"]
        print(f"  ... paso {paso}/{n_pasos_esperados}, minuto {t:.0f}")
    if truncated or terminated:
        break

print(f"\n{paso} pasos simulados. Recompensa total: {reward_total:.2f}")


  ... paso 90/540, minuto 540
  ... paso 180/540, minuto 720
  ... paso 270/540, minuto 900
  ... paso 360/540, minuto 1080
  ... paso 450/540, minuto 1260


  ... paso 540/540, minuto 0

540 pasos simulados. Recompensa total: -1202.49


## Verificacion de conservacion de personas

In [3]:
conservacion = met.verificar_conservacion(env)
for k, v in conservacion.items():
    print(f"{k}: {v}")
assert conservacion["cuadra"], "La conservacion de personas no cuadra -- hay un bug que exponer, no esconder."
print("\nOK: todas las personas generadas quedaron contabilizadas (atendidas + esperando al final + a bordo al final). Nadie se pierde -- ver simulacion/README.md.")


generadas: 1106
atendidas: 1106
esperando_al_final: 0
a_bordo_al_final: 0
suma: 1106
cuadra: True

OK: todas las personas generadas quedaron contabilizadas (atendidas + esperando al final + a bordo al final). Nadie se pierde -- ver simulacion/README.md.


## Metricas detalladas

In [4]:
reporte = met.reporte_completo(env)

print("=== Globales ===")
for k, v in reporte["globales"].items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== Por par (solo pares con demanda) ===")
df_par = reporte["por_par"]
display(df_par[df_par["generadas"] > 0].round(1))


=== Globales ===
  unidades_generadas: 1106
  unidades_atendidas: 1106
  unidades_sin_atender_al_final: 0
  pct_atendidas: 100.00
  espera_media_min: 10.85
  sistema_medio_min: 20.52
  sistema_maximo_min: 53.62

=== Por par (solo pares con demanda) ===


,par,generadas,atendidas,sin_atender_al_final,pct_atendidas,espera_media_min,espera_maxima_min,viaje_medio_min,sistema_medio_min,sistema_maximo_min
0,kleppesto->laksevag,102,102,0,100.0,11.2,19.8,6.0,17.2,25.8
1,kleppesto->bryggen,210,210,0,100.0,9.5,27.1,12.0,21.5,39.1
2,kleppesto->sandviken,42,42,0,100.0,22.7,43.6,10.0,32.7,53.6
3,laksevag->kleppesto,77,77,0,100.0,13.0,26.3,6.0,19.0,32.3
4,laksevag->bryggen,87,87,0,100.0,15.4,29.7,10.0,25.4,39.7
5,laksevag->sandviken,22,22,0,100.0,8.1,11.4,10.0,18.1,21.4
6,bryggen->kleppesto,184,184,0,100.0,7.0,20.8,12.0,19.0,32.8
7,bryggen->laksevag,112,112,0,100.0,8.0,29.5,10.0,18.0,39.5
8,bryggen->sandviken,125,125,0,100.0,10.9,25.0,8.0,18.9,33.0
9,sandviken->kleppesto,9,9,0,100.0,15.8,15.8,10.0,25.8,25.8


In [5]:
print("=== Por barco ===")
display(reporte["por_barco"].round(1))

print("\n=== Por usuario (percentiles, minutos) ===")
for k, v in reporte["por_usuario"].items():
    print(f"  {k}: {v}")

print("\n=== Sin atender al final (backlog, no perdidas) ===")
display(reporte["sin_atender_al_final"])

reporte["por_par"].to_csv(OUT_ESC / "metricas_por_par.csv", index=False)
reporte["por_barco"].to_csv(OUT_ESC / "metricas_por_barco.csv", index=False)


=== Por barco ===


,barco_id,movimientos,tiempo_navegado_min,ocupacion_media,ocupacion_maxima,pct_ocioso,ruta_mas_frecuente
0,barco_0,83,642,2.9,20,40.7,kleppesto->bryggen (17x)
1,barco_1,77,586,2.2,20,45.8,bryggen->kleppesto (12x)
2,barco_2,78,580,2.7,20,46.4,bryggen->kleppesto (13x)



=== Por usuario (percentiles, minutos) ===
  espera_min: {'p50': 9.379999999999995, 'p90': 21.600000000000023, 'p95': 26.319999999999993, 'max': 43.620000000000005}
  viaje_min: {'p50': 10.0, 'p90': 12.0, 'p95': 12.0, 'max': 12.0}
  sistema_min: {'p50': 17.99000000000001, 'p90': 32.31999999999999, 'p95': 35.839999999999975, 'max': 53.620000000000005}

=== Sin atender al final (backlog, no perdidas) ===


,par,sin_atender


## Logs crudos (para auditar sin volver a correr nada)

In [6]:
pd.DataFrame(env.log_eventos).to_csv(OUT_ESC / "log_eventos.csv", index=False)
pd.DataFrame(env.log_recompensa).to_csv(OUT_ESC / "log_recompensa.csv", index=False)
print(f"{len(env.log_eventos)} eventos guardados en log_eventos.csv")
print(f"{len(env.log_recompensa)} pasos de recompensa guardados en log_recompensa.csv")


3166 eventos guardados en log_eventos.csv
540 pasos de recompensa guardados en log_recompensa.csv


## Graficas

In [7]:
fig = viz.graficar_perfil_espera(env)
fig.write_html(OUT_ESC / "wait_profile.html")
fig.show()


In [8]:
fig = viz.graficar_ocupacion_flota(env)
fig.write_html(OUT_ESC / "fleet_occupancy.html")
fig.show()


In [9]:
fig = viz.graficar_heatmap_cumplimiento(env)
fig.write_html(OUT_ESC / "pct_served_heatmap.html")
fig.show()


In [10]:
fig = viz.graficar_desglose_recompensa(env)
fig.write_html(OUT_ESC / "reward_breakdown.html")
fig.show()


In [11]:
fig = viz.graficar_sin_atender_al_final(env)
fig.write_html(OUT_ESC / "backlog_by_pair.html")
fig.show()


## Verificacion de reproducibilidad

In [12]:
sys.path.insert(0, str((BASE_DIR / "../demand/src").resolve()))
import llegadas as demand_llegadas
import masas as demand_masas

DEMAND_DIR = (BASE_DIR / "../demand").resolve()
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")
resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
bb_cfg_path2 = (DEMAND_DIR / cfg_demand["fuentes_externas"]["nodos_config"]).resolve()
cfg_bb2 = yaml.safe_load(open(bb_cfg_path2, encoding="utf-8"))
conexiones_fuertes = cfg_bb2["garantia"]["conexiones_fuertes"]


def generar_grupos_con_semilla(semilla):
    cfg_mod = dict(cfg_demand)
    cfg_mod["demanda"] = dict(cfg_demand["demanda"])
    cfg_mod["demanda"]["porcentaje_poblacion_dia"] = cfg_escalon["porcentaje_poblacion_dia"]
    rng = np.random.default_rng(semilla)
    grupos = demand_llegadas.generar_llegadas_dia(
        cfg_mod, intensidad_od, conexiones_fuertes, poblacion_total_zonas,
        es_fin_de_semana=False, rng=rng, dia_id="repro2",
    )
    hora_ini, hora_fin = cfg_escalon["horas"]
    return grupos[(grupos["hora"] >= hora_ini) & (grupos["hora"] < hora_fin)].reset_index(drop=True)


def correr_una_vez(grupos_df):
    env2 = SimuladorBarcosBergen(
        grupos_df=grupos_df, matriz_tiempos=matriz_tiempos, nodos=nodos,
        num_barcos=cfg_escalon["num_barcos"], capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"],
        nodo_inicial=cfg_bb["flota"]["nodo_inicial"], paso_tiempo_min=cfg_sim["paso_tiempo_min"],
        hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
    )
    obs, info = env2.reset(seed=cfg_sim["semilla"])
    total = 0.0
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones = asignar_flota(libres, estado, matriz_tiempos, env2.capacidad_barco, cfg_sim["recompensa"])
        accion = np.array([
            env2.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
            for b in estado.barcos
        ])
        obs, r, terminated, truncated, info = env2.step(accion)
        total += r
        if truncated or terminated:
            break
    return round(total, 6), len(env2.atendidas_historico), sum(len(c) for c in env2.colas.values())


grupos_semilla_42_a = generar_grupos_con_semilla(42)
grupos_semilla_42_b = generar_grupos_con_semilla(42)
grupos_semilla_43 = generar_grupos_con_semilla(43)

r1 = correr_una_vez(grupos_semilla_42_a)
r2 = correr_una_vez(grupos_semilla_42_b)
r3 = correr_una_vez(grupos_semilla_43)
print("Semilla 42, generacion a:", r1)
print("Semilla 42, generacion b:", r2)
print("Semilla 43:              ", r3)
print("Misma semilla de demanda -> misma corrida completa:", r1 == r2)
print("Semilla distinta -> corrida distinta:", r1 != r3)


Semilla 42, generacion a: (-1202.48656, 1106, 0)
Semilla 42, generacion b: (-1202.48656, 1106, 0)
Semilla 43:               (-675.554602, 908, 0)
Misma semilla de demanda -> misma corrida completa: True
Semilla distinta -> corrida distinta: True
